# Telugu Corpus Collector — 200M Token Target
**Sources**: Wikipedia + Wikisource + Wikibooks + Wiktionary (temp file → process → auto-deleted) + 20 News Sites + Vikaspedia + Internet Archive

- Dumps are downloaded to a temp file, processed, then **automatically deleted**.
- Crash-safe: every source wrapped in try/except. Reconnects resume from checkpoint.


In [1]:
import sys
!{sys.executable} -m pip install -q requests beautifulsoup4 lxml


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import os, re, json, time, bz2, threading, requests, traceback
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, quote
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Environment ────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = '/content/drive/MyDrive/2025201027_LMA_DATASET_TELUGU'
    print('Colab: Drive mounted.')
except ImportError:
    BASE_DIR = os.path.abspath('telugu_dataset_output')
    print('Local mode.')

OUT_DIR    = os.path.join(BASE_DIR, 'web_crawl_output')
os.makedirs(OUT_DIR, exist_ok=True)

TXT_FILE   = os.path.join(OUT_DIR, 'raw_web_crawled_telugu.txt')
CKPT_FILE  = os.path.join(OUT_DIR, 'crawled_urls.json')
FAIL_FILE  = os.path.join(OUT_DIR, 'failed_urls.json')
STATS_FILE = os.path.join(OUT_DIR, 'web_crawl_stats.json')

HEADERS = {'User-Agent': 'TeluguCorpusBot/1.0 (academic)',
           'Accept-Language': 'te,en;q=0.8'}

state_lock = threading.Lock()
file_lock  = threading.Lock()
domain_ts  = {}
_save_ctr  = 0

# ── Checkpoint helpers ─────────────────────────────────────────
def load_ckpt():
    try:
        done = set(json.load(open(CKPT_FILE, encoding='utf-8'))) if os.path.exists(CKPT_FILE) else set()
    except Exception:
        done = set()
    try:
        fail = set(json.load(open(FAIL_FILE, encoding='utf-8'))) if os.path.exists(FAIL_FILE) else set()
    except Exception:
        fail = set()
    return done, fail

def flush_ckpt(done, fail, stats):
    with state_lock:
        try: json.dump(sorted(done),  open(CKPT_FILE,  'w', encoding='utf-8'), ensure_ascii=False)
        except Exception: pass
        try: json.dump(sorted(fail),  open(FAIL_FILE,  'w', encoding='utf-8'), ensure_ascii=False)
        except Exception: pass
        try: json.dump(stats,         open(STATS_FILE, 'w', encoding='utf-8'), ensure_ascii=False)
        except Exception: pass

def load_stats():
    try:
        return json.load(open(STATS_FILE, encoding='utf-8')) if os.path.exists(STATS_FILE) else {}
    except Exception:
        return {}

def est_tokens(stats):
    return int(sum(s.get('w', 0) for s in stats.values()) * 1.7)

# ── Rate limiter (per domain) ──────────────────────────────────
def throttle(url, delay=0.15):
    d = urlparse(url).netloc
    with state_lock:
        wait = delay - (time.time() - domain_ts.get(d, 0))
        if wait > 0: time.sleep(wait)
        domain_ts[d] = time.time()

# ── Telugu text quality filter ─────────────────────────────────
def te_ratio(txt):
    if not txt: return 0.0
    te = sum(1 for c in txt if '\u0C00' <= c <= '\u0C7F')
    ns = sum(1 for c in txt if not c.isspace())
    return te / ns if ns else 0.0

def clean_wiki(txt):
    if not txt: return ''
    txt = re.sub(r'<!--.*?-->',                          '',  txt, flags=re.DOTALL)
    txt = re.sub(r'<ref[^>]*>.*?</ref>',                 '',  txt, flags=re.DOTALL)
    txt = re.sub(r'<ref[^>]*/?>',                        '',  txt)
    txt = re.sub(r'\{\{.*?\}\}',                         '',  txt, flags=re.DOTALL)
    txt = re.sub(r'\[\[(File|Image|చిత్రం):.*?\]\]',     '',  txt, flags=re.I|re.DOTALL)
    txt = re.sub(r'\[\[[^\]|]*\|([^\]]+)\]\]',           r'\1', txt)
    txt = re.sub(r'\[\[([^\]]+)\]\]',                    r'\1', txt)
    txt = re.sub(r'==+[^=]+==+|<[^>]+>|\[http[^\]]*\]', '',  txt)
    lines = [re.sub(r'\s+', ' ', l).strip() for l in txt.split('\n')]
    return '\n'.join(l for l in lines if len(l) >= 20 and te_ratio(l) >= 0.45)

def clean_html(html):
    try: soup = BeautifulSoup(html, 'lxml')
    except Exception: soup = BeautifulSoup(html, 'html.parser')
    for t in soup(['script','style','header','footer','nav','aside','noscript','iframe']):
        t.decompose()
    body = soup.find('main') or soup.find('article') or soup.body or soup
    lines = []
    for p in body.find_all(['p','h1','h2','h3','li']):
        l = re.sub(r'\s+', ' ', p.get_text(' ')).strip()
        if len(l) >= 20 and te_ratio(l) >= 0.50:
            lines.append(l)
    return '\n'.join(lines)

# ── Write one document (thread-safe, atomic checkpoint every 25) 
def write_doc(src, uid, text, done, fail, stats):
    global _save_ctr
    wc = len(text.split())
    try:
        with file_lock:
            with open(TXT_FILE, 'a', encoding='utf-8') as f:
                f.write(f'<<SRC:{src}>>\n<<ID:{uid}>>\n{text}\n<<END>>\n\n')
    except Exception as e:
        print(f'  [write_doc] error: {e}'); return 0
    with state_lock:
        done.add(uid)
        stats[uid] = {'w': wc}
        _save_ctr += 1
        if _save_ctr % 25 == 0:
            try: json.dump(sorted(done),  open(CKPT_FILE,  'w', encoding='utf-8'), ensure_ascii=False)
            except Exception: pass
            try: json.dump(sorted(fail),  open(FAIL_FILE,  'w', encoding='utf-8'), ensure_ascii=False)
            except Exception: pass
            try: json.dump(stats,         open(STATS_FILE, 'w', encoding='utf-8'), ensure_ascii=False)
            except Exception: pass
    return wc

print('✅ Setup complete.')
print(f'Output folder : {OUT_DIR}')
print(f'Text file     : {TXT_FILE}')
done0, _ = load_ckpt(); st0 = load_stats()
print(f'Already saved : {len(done0):,} pages | ~{est_tokens(st0):,} tokens')


Local mode.
✅ Setup complete.
Output folder : /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output
Text file     : /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/raw_web_crawled_telugu.txt
Already saved : 22,165 pages | ~0 tokens


/Users/erva_rishitha/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# ── WIKIMEDIA DUMPS ────────────────────────────────────────────
# Download → process → auto-delete temp file. No permanent .bz2.
# If download is interrupted, temp file is cleaned up automatically.

DUMPS = [
    ('Telugu_Wikipedia',
     'https://dumps.wikimedia.org/tewiki/latest/tewiki-latest-pages-articles.xml.bz2',
     'https://te.wikipedia.org/wiki/'),
    ('Telugu_Wikisource',
     'https://dumps.wikimedia.org/tewikisource/latest/tewikisource-latest-pages-articles.xml.bz2',
     'https://te.wikisource.org/wiki/'),
    ('Telugu_Wikibooks',
     'https://dumps.wikimedia.org/tewikibooks/latest/tewikibooks-latest-pages-articles.xml.bz2',
     'https://te.wikibooks.org/wiki/'),
    ('Telugu_Wiktionary',
     'https://dumps.wikimedia.org/tewiktionary/latest/tewiktionary-latest-pages-articles.xml.bz2',
     'https://te.wiktionary.org/wiki/'),
]

def process_dump(name, url, base_url):
    done, fail = load_ckpt()
    stats = load_stats()
    tmp = os.path.join(OUT_DIR, f'_tmp_{name}.xml.bz2')

    print(f'\n── {name} ──')
    print(f'   URL     : {url}')
    print(f'   Temp    : {tmp}  (auto-deleted after parsing)')

    # ── Download ──────────────────────────────────────────────
    try:
        with requests.get(url, headers=HEADERS, stream=True, timeout=120) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            dl = 0
            with open(tmp, 'wb') as f:
                for chunk in r.iter_content(chunk_size=4*1024*1024):
                    if chunk:
                        f.write(chunk)
                        dl += len(chunk)
                        pct = f' ({dl/total*100:.0f}%)' if total else ''
                        print(f'\r   Downloading: {dl/1024/1024:.0f} MB{pct}', end='', flush=True)
        print(f'\n   Download done ({dl/1024/1024:.0f} MB). Parsing...')
    except Exception as e:
        print(f'\n   ❌ Download failed: {e}')
        if os.path.exists(tmp): os.remove(tmp)
        return

    # ── Parse (streaming XML — no extra RAM) ──────────────────
    pages, words = 0, 0
    try:
        with bz2.BZ2File(tmp, 'rb') as bz:
            for event, elem in ET.iterparse(bz, events=('end',)):
                try:
                    if not elem.tag.endswith('page'):
                        continue
                    ns = elem.find('{*}ns')
                    if ns is not None and ns.text != '0':
                        elem.clear(); continue
                    title_el = elem.find('{*}title')
                    title    = title_el.text if title_el is not None else ''
                    uid      = f"{base_url}{quote(title.replace(' ', '_'))}"
                    if uid in done:
                        elem.clear(); continue
                    rev = elem.find('{*}revision')
                    if rev is not None:
                        te_el = rev.find('{*}text')
                        wt    = te_el.text if te_el is not None else ''
                        if wt and wt.strip().upper().startswith('#REDIRECT'):
                            elem.clear(); continue
                        txt = clean_wiki(wt)
                        if len(txt.split()) > 30:
                            write_doc(name, uid, txt, done, fail, stats)
                            pages += 1; words += len(txt.split())
                            if pages % 2000 == 0:
                                flush_ckpt(done, fail, stats)
                                print(f'   {pages:,} articles | +{words:,} words | ~{est_tokens(stats):,} tokens total')
                except Exception:
                    pass  # skip any single bad article, never crash
                finally:
                    elem.clear()
    except Exception as e:
        print(f'   ❌ Parse error: {e}')
    finally:
        # Always delete temp file
        if os.path.exists(tmp):
            os.remove(tmp)
            print(f'   🗑️  Temp file deleted: {tmp}')

    flush_ckpt(done, fail, stats)
    print(f'   ✅ {name}: {pages:,} articles | +{words:,} words | ~{est_tokens(stats):,} tokens')


for name, url, base in DUMPS:
    try:
        process_dump(name, url, base)
    except Exception as e:
        print(f'\n❌ {name} crashed (skipping): {e}')
        traceback.print_exc()



── Telugu_Wikipedia ──
   URL     : https://dumps.wikimedia.org/tewiki/latest/tewiki-latest-pages-articles.xml.bz2
   Temp    : /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/_tmp_Telugu_Wikipedia.xml.bz2  (auto-deleted after parsing)
   Downloading: 172 MB (70%)
   ❌ Download failed: ('Connection broken: IncompleteRead(181272201 bytes read, 77077789 more expected)', IncompleteRead(181272201 bytes read, 77077789 more expected))

── Telugu_Wikisource ──
   URL     : https://dumps.wikimedia.org/tewikisource/latest/tewikisource-latest-pages-articles.xml.bz2
   Temp    : /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/_tmp_Telugu_Wikisource.xml.bz2  (auto-deleted after parsing)
   Downloading: 113 MB (100%)
   Download done (113 MB). Parsing...
   🗑️  Temp file deleted: /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/_tmp_Telugu_Wikisource.xml.bz2
   ✅ Te

In [4]:
# ── NEWS SITES — 20 Telugu newspapers via sitemap + RSS ────────

FEEDS = [
    ('https://www.sakshi.com/sitemap.xml',           'sitemap'),
    ('https://www.sakshi.com/rss.xml',               'rss'),
    ('https://www.ntnews.com/sitemap.xml',           'sitemap'),
    ('https://www.eenadu.net/sitemap.xml',           'sitemap'),
    ('https://www.eenadu.net/rss/telugu-news.xml',   'rss'),
    ('https://www.andhrajyothy.com/sitemap.xml',     'sitemap'),
    ('https://www.andhrajyothy.com/rss',             'rss'),
    ('https://www.prajasakti.com/sitemap.xml',       'sitemap'),
    ('https://www.suryaa.com/sitemap.xml',           'sitemap'),
    ('https://www.andhrabhoomi.net/sitemap.xml',     'sitemap'),
    ('https://www.navatelangana.com/sitemap.xml',    'sitemap'),
    ('https://telanganatoday.com/sitemap.xml',       'sitemap'),
    ('https://10tv.in/sitemap.xml',                  'sitemap'),
    ('https://telugu.greatandhra.com/sitemap.xml',   'sitemap'),
    ('https://telugu.oneindia.com/sitemap.xml',      'sitemap'),
    ('https://tv9telugu.com/sitemap.xml',            'sitemap'),
    ('https://www.bbc.com/telugu/index.xml',         'rss'),
    ('https://telugu.samayam.com/sitemap.cms',       'sitemap'),
    ('https://telugu.webdunia.com/sitemap.xml',      'sitemap'),
    ('https://www.vaartha.com/feed/',                'rss'),
    ('https://telugu.boldsky.com/sitemap.xml',       'sitemap'),
    ('https://www.thehansindia.com/sitemap.xml',     'sitemap'),
]

def get_urls(session, url, ftype, done, depth=0):
    """Recursively expand sitemapindex → sub-sitemaps → article URLs."""
    if depth > 2: return []
    try:
        throttle(url)
        r = session.get(url, headers=HEADERS, timeout=12)
        if r.status_code != 200: return []
        try: soup = BeautifulSoup(r.content, 'lxml-xml')
        except Exception: soup = BeautifulSoup(r.content, 'html.parser')
        if ftype == 'rss':
            out = []
            for item in soup.find_all(['item','entry']):
                lk = item.find('link')
                if not lk: continue
                u = (lk.get('href') or lk.text or '').strip()
                if u.startswith('http') and u not in done: out.append(u)
            return out
        # sitemap
        subs = soup.find_all('sitemap')
        if subs:
            out = []
            for sm in subs:
                loc = sm.find('loc')
                if loc and loc.text.strip().startswith('http'):
                    out += get_urls(session, loc.text.strip(), 'sitemap', done, depth+1)
            return out
        return [l.text.strip() for l in soup.find_all('loc')
                if l.text.strip().startswith('http')
                and not any(l.text.strip().endswith(x) for x in ['.xml','.jpg','.png','.gif'])
                and l.text.strip() not in done]
    except Exception:
        return []

def fetch_article(session, url, done, fail, stats):
    if url in done or url in fail: return 0
    try:
        throttle(url)
        r = session.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 404:
            with state_lock: fail.add(url)
            return 0
        if r.status_code in (429, 503):
            time.sleep(5); return 0
        if r.status_code != 200: return 0
        txt = clean_html(r.content)
        if len(txt.split()) > 50:
            return write_doc('news', url, txt, done, fail, stats)
    except Exception:
        pass
    return 0

done, fail = load_ckpt()
stats = load_stats()
session = requests.Session()
session.max_redirects = 5

print('=== News Crawl (20 sources) ===')
for feed_url, ftype in FEEDS:
    try:
        urls = list(set(get_urls(session, feed_url, ftype, done)))
        urls = [u for u in urls if u not in done]
        print(f'  {feed_url[:60]}\n    → {len(urls):,} new URLs')
        if not urls: continue
        saved = 0
        with ThreadPoolExecutor(max_workers=6) as ex:
            futs = [ex.submit(fetch_article, session, u, done, fail, stats) for u in urls]
            for fut in as_completed(futs):
                try:
                    if fut.result() > 0: saved += 1
                except Exception:
                    pass
        flush_ckpt(done, fail, stats)
        print(f'    → saved {saved:,} | ~{est_tokens(stats):,} tokens')
    except Exception as e:
        print(f'  ❌ {feed_url}: {e} (skipped)')

flush_ckpt(done, fail, stats)
print(f'\nNews done. Total tokens: ~{est_tokens(stats):,}')


=== News Crawl (20 sources) ===
  https://www.sakshi.com/sitemap.xml
    → 30 new URLs
    → saved 17 | ~16,049 tokens
  https://www.sakshi.com/rss.xml
    → 0 new URLs
  https://www.ntnews.com/sitemap.xml
    → 0 new URLs
  https://www.eenadu.net/sitemap.xml
    → 4,270 new URLs
    → saved 4,191 | ~5,976,290 tokens
  https://www.eenadu.net/rss/telugu-news.xml
    → 0 new URLs
  https://www.andhrajyothy.com/sitemap.xml
    → 0 new URLs
  https://www.andhrajyothy.com/rss
    → 0 new URLs
  https://www.prajasakti.com/sitemap.xml
    → 0 new URLs
  https://www.suryaa.com/sitemap.xml
    → 19 new URLs
    → saved 1 | ~5,977,174 tokens
  https://www.andhrabhoomi.net/sitemap.xml
    → 0 new URLs
  https://www.navatelangana.com/sitemap.xml
    → 0 new URLs
  https://telanganatoday.com/sitemap.xml
    → 63 new URLs
    → saved 0 | ~5,977,174 tokens
  https://10tv.in/sitemap.xml
    → 0 new URLs
  https://telugu.greatandhra.com/sitemap.xml
    → 87,784 new URLs
    → saved 76,221 | ~53,716,198

In [5]:
# ── VIKASPEDIA PORTALS ─────────────────────────────────────────

PORTALS = [
    'https://te.vikaspedia.in/agriculture',
    'https://te.vikaspedia.in/health',
    'https://te.vikaspedia.in/education',
    'https://te.vikaspedia.in/social-welfare',
    'https://te.vikaspedia.in/e-governance',
    'https://te.vikaspedia.in/energy',
    'https://te.vikaspedia.in/culture',
    'https://te.vikaspedia.in/science',
    'https://te.vikaspedia.in/livelihood',
    'https://te.vikaspedia.in/disaster-management',
]

done, fail = load_ckpt()
stats = load_stats()
session = requests.Session()

print('=== Vikaspedia Portals ===')
for portal in PORTALS:
    try:
        throttle(portal)
        r = session.get(portal, headers=HEADERS, timeout=12)
        if r.status_code != 200:
            print(f'  Skipping {portal} (HTTP {r.status_code})'); continue
        soup = BeautifulSoup(r.content, 'html.parser')
        links = list(set(
            urljoin(portal, a['href'])
            for a in soup.find_all('a', href=True)
            if urljoin(portal, a['href']).startswith('https://te.vikaspedia.in')
            and '#' not in a['href']
        ))
        links = [u for u in links if u not in done]
        print(f'  {portal[-35:]} → {len(links):,} pages')
        saved = 0
        with ThreadPoolExecutor(max_workers=4) as ex:
            futs = [ex.submit(fetch_article, session, u, done, fail, stats) for u in links]
            for fut in as_completed(futs):
                try:
                    if fut.result() > 0: saved += 1
                except Exception:
                    pass
        flush_ckpt(done, fail, stats)
        print(f'    → saved {saved:,} | ~{est_tokens(stats):,} tokens')
    except Exception as e:
        print(f'  ❌ {portal}: {e} (skipped)')

flush_ckpt(done, fail, stats)
print(f'Portals done. Total tokens: ~{est_tokens(stats):,}')


=== Vikaspedia Portals ===
  ttps://te.vikaspedia.in/agriculture → 0 pages
    → saved 0 | ~53,724,697 tokens
  https://te.vikaspedia.in/health → 0 pages
    → saved 0 | ~53,724,697 tokens
  https://te.vikaspedia.in/education → 0 pages
    → saved 0 | ~53,724,697 tokens
  s://te.vikaspedia.in/social-welfare → 0 pages
    → saved 0 | ~53,724,697 tokens
  tps://te.vikaspedia.in/e-governance → 0 pages
    → saved 0 | ~53,724,697 tokens
  https://te.vikaspedia.in/energy → 0 pages
    → saved 0 | ~53,724,697 tokens
  Skipping https://te.vikaspedia.in/culture (HTTP 404)
  Skipping https://te.vikaspedia.in/science (HTTP 404)
  Skipping https://te.vikaspedia.in/livelihood (HTTP 404)
  Skipping https://te.vikaspedia.in/disaster-management (HTTP 404)
Portals done. Total tokens: ~53,724,697


In [6]:
# ── INTERNET ARCHIVE — open-access Telugu books ────────────────

QUERIES = [
    'language:Telugu AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Telugu literature" AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Telugu" AND subject:"novel" AND mediatype:texts AND NOT access-restricted-item:true',
]
API = 'https://archive.org/advancedsearch.php'

done, fail = load_ckpt()
stats = load_stats()
session = requests.Session()
saved_total = 0

print('=== Internet Archive ===')
for q in QUERIES:
    print(f'  Query: {q[:65]}...')
    try:
        resp = session.get(API, params={
            'q': q, 'fl[]': 'identifier,title',
            'rows': 200, 'page': 1, 'output': 'json', 'sort[]': 'downloads desc'
        }, timeout=20)
        resp.raise_for_status()
        items = resp.json().get('response', {}).get('docs', [])
        print(f'  Found {len(items)} items.')
        for item in items:
            iid   = item.get('identifier', '')
            title = item.get('title', iid)
            key   = f'archive_{iid}'
            if key in done or key in fail: continue
            text = ''
            for tu in [
                f'https://archive.org/download/{iid}/{iid}.txt',
                f'https://archive.org/stream/{iid}/{iid}_djvu.txt',
            ]:
                try:
                    throttle(tu, delay=0.4)
                    rr = session.get(tu, headers=HEADERS, timeout=20)
                    if rr.status_code == 200 and len(rr.text) > 500:
                        text = rr.text; break
                except Exception:
                    pass
            if not text:
                with state_lock: fail.add(key)
                continue
            lines = [l.strip() for l in text.split('\n')
                     if len(l.strip()) >= 20 and te_ratio(l.strip()) >= 0.40]
            clean = '\n'.join(lines)
            if len(clean.split()) > 100:
                write_doc('archive', key, clean, done, fail, stats)
                saved_total += 1
                if saved_total % 20 == 0:
                    flush_ckpt(done, fail, stats)
                    print(f'  Saved {saved_total} books | ~{est_tokens(stats):,} tokens')
    except Exception as e:
        print(f'  ❌ Query error: {e} (continuing)')

flush_ckpt(done, fail, stats)
print(f'Archive done. Saved {saved_total} books. Total tokens: ~{est_tokens(stats):,}')


=== Internet Archive ===
  Query: language:Telugu AND mediatype:texts AND NOT access-restricted-ite...
  Found 200 items.
  Saved 20 books | ~56,622,070 tokens
  Saved 40 books | ~59,211,486 tokens
  Saved 60 books | ~61,323,841 tokens
  Saved 80 books | ~63,238,371 tokens
  Saved 100 books | ~66,459,944 tokens
  Saved 120 books | ~68,476,664 tokens
  Query: subject:"Telugu literature" AND mediatype:texts AND NOT access-re...
  Found 131 items.
  Saved 140 books | ~71,003,563 tokens
  Query: subject:"Telugu" AND subject:"novel" AND mediatype:texts AND NOT ...
  Found 27 items.
Archive done. Saved 147 books. Total tokens: ~71,147,873


In [7]:
# ── FINAL SUMMARY ──────────────────────────────────────────────
stats       = load_stats()
done, fail  = load_ckpt()
total_words = sum(s.get('w', 0) for s in stats.values())
total_tok   = int(total_words * 1.7)
target      = 200_000_000

print('=' * 58)
print('  CORPUS SUMMARY')
print('=' * 58)
print(f'  Unique pages / items saved  : {len(done):,}')
print(f'  Failed / skipped URLs       : {len(fail):,}')
print(f'  Total clean Telugu words    : {total_words:,}')
print(f'  Estimated tokens            : {total_tok:,}')
print(f'  Target                      : {target:,}')
print(f'  Progress toward 200M target : {total_tok/target*100:.1f}%')
if os.path.exists(TXT_FILE):
    mb = os.path.getsize(TXT_FILE) / 1024 / 1024
    print(f'  Output file size            : {mb:.1f} MB')
print(f'  Output file                 : {TXT_FILE}')
print('=' * 58)


  CORPUS SUMMARY
  Unique pages / items saved  : 102,760
  Failed / skipped URLs       : 1,088
  Total clean Telugu words    : 41,851,690
  Estimated tokens            : 71,147,873
  Target                      : 200,000,000
  Progress toward 200M target : 35.6%
  Output file size            : 1020.0 MB
  Output file                 : /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/raw_web_crawled_telugu.txt


In [8]:
# ── WIKIMEDIA DUMPS (Resilient Download + Namespace Fix) ────────

DUMPS = [
    ('Telugu_Wikipedia',
     'https://dumps.wikimedia.org/tewiki/latest/tewiki-latest-pages-articles.xml.bz2',
     'https://te.wikipedia.org/wiki/', [0]),
    ('Telugu_Wikisource',
     'https://dumps.wikimedia.org/tewikisource/latest/tewikisource-latest-pages-articles.xml.bz2',
     'https://te.wikisource.org/wiki/', [0, 104]), # Includes ns=104 for raw book pages
    ('Telugu_Wikibooks',
     'https://dumps.wikimedia.org/tewikibooks/latest/tewikibooks-latest-pages-articles.xml.bz2',
     'https://te.wikibooks.org/wiki/', [0]),
    ('Telugu_Wiktionary',
     'https://dumps.wikimedia.org/tewiktionary/latest/tewiktionary-latest-pages-articles.xml.bz2',
     'https://te.wiktionary.org/wiki/', [0]),
]

def download_file_with_resume(url, dest_path):
    """Downloads files with automatic chunked retry and range resume."""
    file_mode = 'ab' if os.path.exists(dest_path) else 'wb'
    existing_size = os.path.getsize(dest_path) if os.path.exists(dest_path) else 0
    
    headers = HEADERS.copy()
    if existing_size > 0:
        headers['Range'] = f"bytes={existing_size}-"

    for attempt in range(5):
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                if r.status_code in (200, 206):
                    total = int(r.headers.get('content-length', 0)) + existing_size
                    dl = existing_size
                    with open(dest_path, file_mode) as f:
                        for chunk in r.iter_content(chunk_size=4*1024*1024):
                            if chunk:
                                f.write(chunk)
                                dl += len(chunk)
                                pct = f' ({dl/total*100:.0f}%)' if total else ''
                                print(f'\r   Downloading: {dl/1024/1024:.1f} MB{pct}', end='', flush=True)
                    print(f'\n   Download completed ({dl/1024/1024:.1f} MB).')
                    return True
        except Exception as e:
            print(f"\n   [Retry {attempt+1}/5] Connection drop: {e}. Resuming...")
            time.sleep(3)
            existing_size = os.path.getsize(dest_path) if os.path.exists(dest_path) else 0
            headers['Range'] = f"bytes={existing_size}-"
            file_mode = 'ab'
    return False

def process_dump_fixed(name, url, base_url, valid_ns):
    done, fail = load_ckpt()
    stats = load_stats()
    tmp = os.path.join(OUT_DIR, f'_tmp_{name}.xml.bz2')

    print(f'\n── {name} ──')
    print(f'   URL     : {url}')
    print(f'   Temp    : {tmp}')

    if not download_file_with_resume(url, tmp):
        print(f'   ❌ Could not download {name} after retries.')
        if os.path.exists(tmp): os.remove(tmp)
        return

    pages, words = 0, 0
    min_words = 5 if 'Wiktionary' in name else 25 # Allow shorter dictionary definitions

    try:
        with bz2.BZ2File(tmp, 'rb') as bz:
            for event, elem in ET.iterparse(bz, events=('end',)):
                try:
                    if not elem.tag.endswith('page'):
                        continue
                    ns_el = elem.find('{*}ns')
                    ns_val = int(ns_el.text) if (ns_el is not None and ns_el.text is not None) else 0
                    
                    if ns_val not in valid_ns:
                        elem.clear(); continue

                    title_el = elem.find('{*}title')
                    title    = title_el.text if title_el is not None else ''
                    uid      = f"{base_url}{quote(title.replace(' ', '_'))}"

                    if uid in done:
                        elem.clear(); continue

                    rev = elem.find('{*}revision')
                    if rev is not None:
                        te_el = rev.find('{*}text')
                        wt    = te_el.text if te_el is not None else ''
                        if wt and wt.strip().upper().startswith('#REDIRECT'):
                            elem.clear(); continue
                        
                        txt = clean_wiki(wt)
                        if len(txt.split()) >= min_words:
                            write_doc(name, uid, txt, done, fail, stats)
                            pages += 1
                            words += len(txt.split())
                            if pages % 2000 == 0:
                                flush_ckpt(done, fail, stats)
                                print(f'   {pages:,} articles | +{words:,} words | ~{est_tokens(stats):,} total tokens')
                except Exception:
                    pass
                finally:
                    elem.clear()
    except Exception as e:
        print(f'   ❌ Parse error on {name}: {e}')
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)
            print(f'   🗑️ Temp file deleted: {tmp}')

    flush_ckpt(done, fail, stats)
    print(f'   ✅ {name}: {pages:,} articles | +{words:,} words | ~{est_tokens(stats):,} tokens')

for name, url, base, ns_list in DUMPS:
    try:
        process_dump_fixed(name, url, base, ns_list)
    except Exception as e:
        print(f'\n❌ {name} execution failed: {e}')


── Telugu_Wikipedia ──
   URL     : https://dumps.wikimedia.org/tewiki/latest/tewiki-latest-pages-articles.xml.bz2
   Temp    : /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/_tmp_Telugu_Wikipedia.xml.bz2
   Downloading: 246.4 MB (100%)
   Download completed (246.4 MB).
   🗑️ Temp file deleted: /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/_tmp_Telugu_Wikipedia.xml.bz2
   ✅ Telugu_Wikipedia: 0 articles | +0 words | ~71,147,873 tokens

── Telugu_Wikisource ──
   URL     : https://dumps.wikimedia.org/tewikisource/latest/tewikisource-latest-pages-articles.xml.bz2
   Temp    : /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/_tmp_Telugu_Wikisource.xml.bz2
   Downloading: 113.4 MB (100%)
   Download completed (113.4 MB).
   🗑️ Temp file deleted: /Users/erva_rishitha/Documents/LMA project1/notebooks/telugu_dataset_output/web_crawl_output/_tmp_Telugu_Wikiso

In [9]:
# ── INTERNET ARCHIVE (Deep Paginated Fetching) ────────────────

IA_QUERIES = [
    'language:Telugu AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Telugu literature" AND mediatype:texts AND NOT access-restricted-item:true',
    'subject:"Telugu" AND mediatype:texts AND NOT access-restricted-item:true',
    'title:"Telugu" AND mediatype:texts AND NOT access-restricted-item:true'
]

done, fail = load_ckpt()
stats = load_stats()
session = requests.Session()
saved_total = 0

print('=== Internet Archive Deep Extraction ===')
for q in IA_QUERIES:
    print(f'  Querying: {q[:60]}...')
    for page in range(1, 15):  # Paginate up to 15 pages (3,000 potential books per query)
        try:
            resp = session.get('https://archive.org/advancedsearch.php', params={
                'q': q,
                'fl[]': 'identifier,title',
                'rows': 200,
                'page': page,
                'output': 'json',
                'sort[]': 'downloads desc'
            }, timeout=20)
            
            if resp.status_code != 200: break
            items = resp.json().get('response', {}).get('docs', [])
            if not items: break

            print(f'  Page {page}: Processing {len(items)} metadata records...')
            for item in items:
                iid = item.get('identifier', '')
                key = f'archive_{iid}'
                if key in done or key in fail: continue

                text = ''
                for tu in [
                    f'https://archive.org/download/{iid}/{iid}_djvu.txt',
                    f'https://archive.org/download/{iid}/{iid}.txt'
                ]:
                    try:
                        throttle(tu, delay=0.2)
                        rr = session.get(tu, headers=HEADERS, timeout=15)
                        if rr.status_code == 200 and len(rr.text) > 500:
                            text = rr.text
                            break
                    except Exception:
                        pass

                if not text:
                    with state_lock: fail.add(key)
                    continue

                lines = [l.strip() for l in text.split('\n')
                         if len(l.strip()) >= 20 and te_ratio(l.strip()) >= 0.40]
                clean = '\n'.join(lines)
                
                if len(clean.split()) > 80:
                    write_doc('archive', key, clean, done, fail, stats)
                    saved_total += 1
                    if saved_total % 25 == 0:
                        flush_ckpt(done, fail, stats)
                        print(f'   Saved {saved_total} books | ~{est_tokens(stats):,} total tokens')
        except Exception as e:
            print(f'  ❌ Error on page {page}: {e}')
            break

flush_ckpt(done, fail, stats)
print(f'Archive complete. Total added books: {saved_total}. Total tokens: ~{est_tokens(stats):,}')

=== Internet Archive Deep Extraction ===
  Querying: language:Telugu AND mediatype:texts AND NOT access-restricte...
  Page 1: Processing 200 metadata records...
  ❌ Error on page 2: HTTPSConnectionPool(host='archive.org', port=443): Max retries exceeded with url: /advancedsearch.php?q=language%3ATelugu+AND+mediatype%3Atexts+AND+NOT+access-restricted-item%3Atrue&fl%5B%5D=identifier%2Ctitle&rows=200&page=2&output=json&sort%5B%5D=downloads+desc (Caused by NameResolutionError("HTTPSConnection(host='archive.org', port=443): Failed to resolve 'archive.org' ([Errno 8] nodename nor servname provided, or not known)"))
  Querying: subject:"Telugu literature" AND mediatype:texts AND NOT acce...
  ❌ Error on page 1: HTTPSConnectionPool(host='archive.org', port=443): Max retries exceeded with url: /advancedsearch.php?q=subject%3A%22Telugu+literature%22+AND+mediatype%3Atexts+AND+NOT+access-restricted-item%3Atrue&fl%5B%5D=identifier%2Ctitle&rows=200&page=1&output=json&sort%5B%5D=downloads+desc (Caus